[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/03_Initializers_and_Attributes/Initializers_and_Attributes_Deep_Dive.ipynb)

# 1.3 Initializers and Attributes — Deep Dive

Embed **constant weights** (initializers) and set **fixed operator parameters** (attributes) in ONNX graphs. This transforms a "calculator" graph that requires all values at runtime into a **self-contained model** that carries its learned parameters.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [The Problem: Externalised Weights](#section-1) | Why passing weights as inputs is impractical |
| 2 | [Initializers — Formal Definition](#section-2) | TensorProto constants baked into the graph |
| 3 | [Building a Model with Initializers](#section-3) | Step-by-step code walkthrough |
| 4 | [Initializer Resolution Rules](#section-4) | What happens when a name appears in both inputs and initializers |
| 5 | [Attributes — Formal Definition](#section-5) | Compile-time operator parameters |
| 6 | [Building a Model with Attributes](#section-6) | Transpose example: $Y = XA^\top + B$ |
| 7 | [Inputs vs Initializers vs Attributes](#section-7) | The three-way taxonomy |
| 8 | [Tensor Shapes and Broadcasting](#section-8) | Broadcasting rules in ONNX |
| 9 | [Numerical Walkthrough](#section-9) | Tracing values through an initializer-based model |
| 10 | [Visualizing the Graph](#section-10) | Matplotlib diagram of data flow |
| 11 | [Advanced: Multiple Data Types](#section-11) | Initializers with different dtypes |
| 12 | [Key Takeaways & Interview Questions](#section-12) | Summary and self-test |

### Prerequisites

- Completed **1.1 Linear Regression** (graph construction basics)
- Completed **1.2 Serialization** (save / load mechanics)
- Familiarity with NumPy arrays and basic linear algebra

<a id='section-1'></a>
## Section 1: The Problem — Externalised Weights

### Recap: The Fully-Dynamic Graph

In notebook 1.1 we built a linear regression graph $Y = XA + B$ where **all three tensors** ($X$, $A$, $B$) were declared as graph inputs. At inference time, the caller had to supply:

```python
session.run(None, {'X': x_data, 'A': weight_data, 'B': bias_data})
```

This is equivalent to a function signature with **no defaults**:

$$f(X, A, B) = XA + B$$

### Why This Is Impractical

In a real deployment scenario, the weights $A \in \mathbb{R}^{D \times K}$ and bias $B \in \mathbb{R}^{K}$ are **learned during training** and should be **fixed constants** inside the model. Only the input $X$ changes at runtime:

$$f_{A,B}(X) = XA + B \quad \text{where } A, B \text{ are frozen}$$

| Approach | Runtime Feeds | Model Self-Contained? | Practical? |
|----------|:------------:|:--------------------:|:----------:|
| All inputs dynamic | $X, A, B$ | No | No — weights must be managed externally |
| Weights as initializers | $X$ only | Yes | Yes — standard production pattern |

### The Solution: Initializers

ONNX solves this with **initializers** — `TensorProto` objects embedded directly in the `GraphProto`. They provide **default values** for tensor names, so the runtime uses the stored data without requiring the caller to supply it.

```
BEFORE (all dynamic):              AFTER (with initializers):

  Caller must provide:              Caller provides only:
  ┌───┐  ┌───┐  ┌───┐              ┌───┐
  │ X │  │ A │  │ B │              │ X │
  └─┬─┘  └─┬─┘  └─┬─┘              └─┬─┘
    │      │      │                    │      ┌───────────────┐
    ▼      ▼      │                    │      │ Stored in     │
  ┌──────────┐    │                    │      │ model file:   │
  │  MatMul  │    │                    ▼      │  A  (init)    │
  └────┬─────┘    │                ┌──────┐   │  B  (init)    │
       │          ▼                │MatMul│◀──│               │
       ▼     ┌────────┐           └──┬───┘   └───────────────┘
     ┌─────┐ │        │              │
     │ Add │◀┘        │              ▼
     └──┬──┘          │           ┌─────┐
        ▼             │           │ Add │◀── B (from init)
      ┌───┐           │           └──┬──┘
      │ Y │           │              ▼
      └───┘           │            ┌───┐
                      │            │ Y │
                      │            └───┘
```

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install onnx onnxruntime matplotlib numpy

<a id='section-2'></a>
## Section 2: Initializers — Formal Definition

### Definition 2.1 (Initializer)

An **initializer** is a `TensorProto` stored in the `GraphProto.initializer` repeated field. It associates a **tensor name** with **concrete data** that is available before any node executes.

Formally, an initializer defines a mapping:

$$\text{init}: \text{name} \mapsto T \in \mathbb{T}^{d_1 \times d_2 \times \cdots \times d_r}$$

where $\mathbb{T}$ is the element type (e.g., `float32`) and $(d_1, \ldots, d_r)$ is the shape.

### TensorProto Structure

```
TensorProto
├── name       : string            ← must match a tensor name in the graph
├── dims       : repeated int64    ← shape (e.g., [2, 3] for a 2×3 matrix)
├── data_type  : int32             ← element type (1=FLOAT, 7=INT64, ...)
├── raw_data   : bytes             ← compact binary storage
└── float_data : repeated float    ← alternative: typed repeated field
```

### Creating Initializers with `numpy_helper`

The `onnx.numpy_helper.from_array()` function converts a NumPy array into a `TensorProto`:

$$\text{numpy.ndarray} \xrightarrow{\text{from\_array}(\cdot, \text{name})} \text{TensorProto}$$

The function automatically:
1. Maps the NumPy dtype to the ONNX `data_type` enum
2. Copies the shape to `dims`
3. Stores the data in `raw_data` (little-endian bytes)
4. Sets the `name` field

### Size Impact

For a weight matrix $W \in \mathbb{R}^{m \times n}$ stored as `float32`:

$$\text{Size}_{\text{bytes}} = m \times n \times 4 \text{ bytes}$$

This is the dominant factor in `.onnx` file size. For example, a single $1000 \times 500$ weight matrix occupies $2{,}000{,}000$ bytes $\approx 1.9$ MB.

<a id='section-3'></a>
## Section 3: Building a Model with Initializers

Let's build the linear regression model $Y = XA + B$ with $A$ and $B$ as initializers.

### Computation Graph

![Linear Regression with Initializers](assets/dot_linreg2.png)

Notice that $A$ and $B$ now flow from the initializer store rather than from external inputs. Only $X$ needs to be provided at runtime.

In [ ]:
import numpy as np
from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info)
from onnx.numpy_helper import from_array, to_array
from onnx.checker import check_model

# ── Step 1: Create initializers from NumPy arrays ─────────────────
# These are the "learned" parameters baked into the model.

A_data = np.array([[0.5, -0.3],
                   [0.8,  0.2]], dtype=np.float32)  # shape (2, 2)

B_data = np.array([0.1, -0.1], dtype=np.float32)    # shape (2,)

A_init = from_array(A_data, name='A')
B_init = from_array(B_data, name='B')

print(f'Initializer A: name={A_init.name!r}, dims={list(A_init.dims)}, '
      f'dtype={TensorProto.DataType.Name(A_init.data_type)}')
print(f'Initializer B: name={B_init.name!r}, dims={list(B_init.dims)}, '
      f'dtype={TensorProto.DataType.Name(B_init.data_type)}')
print(f'\nA raw_data size: {len(A_init.raw_data)} bytes '
      f'(expected: {A_data.size * A_data.itemsize})')
print(f'B raw_data size: {len(B_init.raw_data)} bytes '
      f'(expected: {B_data.size * B_data.itemsize})')

In [ ]:
# ── Step 2: Declare ONLY X as a dynamic input ────────────────────
# A and B are NOT listed as inputs — they come from initializers.

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 2])  # (N, 2)
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 2])  # (N, 2)

# ── Step 3: Create operator nodes ────────────────────────────────
node_matmul = make_node('MatMul', ['X', 'A'], ['XA'])
node_add    = make_node('Add',    ['XA', 'B'], ['Y'])

# ── Step 4: Assemble graph WITH initializers ─────────────────────
# The 5th argument to make_graph is the initializer list.
graph = make_graph(
    [node_matmul, node_add],   # nodes
    'lr_with_initializers',    # graph name
    [X],                       # inputs  (only X!)
    [Y],                       # outputs
    [A_init, B_init]           # initializers
)

model_init = make_model(graph)
check_model(model_init)

print('\nModel built successfully!')
print(f'Graph inputs:       {[i.name for i in model_init.graph.input]}')
print(f'Graph outputs:      {[o.name for o in model_init.graph.output]}')
print(f'Initializers:       {[i.name for i in model_init.graph.initializer]}')
print(f'Nodes:              {[(n.op_type, list(n.input), list(n.output)) for n in model_init.graph.node]}')

In [ ]:
# ── Step 5: Run inference — only X needs to be provided ──────────
import onnxruntime as ort

sess = ort.InferenceSession(
    model_init.SerializeToString(),
    providers=['CPUExecutionProvider'])

x_test = np.array([[1.0, 2.0],
                   [3.0, 4.0],
                   [5.0, 6.0]], dtype=np.float32)

result = sess.run(None, {'X': x_test})[0]

expected = x_test @ A_data + B_data

print('Input X:')
print(x_test)
print('\nONNX Runtime output:')
print(result)
print('\nNumPy verification:')
print(expected)
print(f'\nMatch: {np.allclose(result, expected)}')

<a id='section-4'></a>
## Section 4: Initializer Resolution Rules

### The Three Cases

The ONNX spec defines precise resolution rules for how tensor names are bound to values at runtime. The behavior depends on whether a name appears in **inputs**, **initializers**, or **both**:

| Name appears in... | Behavior | Use Case |
|:---|:---|:---|
| **Initializer only** | Always uses stored value; cannot be overridden | Frozen weights |
| **Both input AND initializer** | Initializer is the **default**; caller can override | Fine-tunable weights |
| **Input only** | Must be provided at runtime | Dynamic data ($X$) |

### Formal Rule

Let $\mathcal{I}$ be the set of input names, $\mathcal{W}$ be the set of initializer names, and $\mathcal{F}$ be the set of names in the runtime feed dictionary. For a tensor name $n$:

$$\text{value}(n) = \begin{cases}
\mathcal{F}[n] & \text{if } n \in \mathcal{F} \text{ (caller provided it)} \\
\mathcal{W}[n] & \text{if } n \in \mathcal{W} \text{ (use initializer default)} \\
\text{ERROR} & \text{otherwise (missing required input)}
\end{cases}$$

### Why "Both" Matters

Listing a name in both inputs and initializers creates an **overridable default**. This pattern is useful for:
- **Transfer learning**: Override pre-trained weights with fine-tuned versions
- **A/B testing**: Swap weight variants without rebuilding the model
- **Debugging**: Inject specific weight values to isolate issues

```
Resolution Priority:

  ┌─────────────────────┐
  │  Feed dict provided  │──── YES ──▶ Use feed value
  │  value for name?     │
  └─────────┬────────────┘
            │ NO
            ▼
  ┌─────────────────────┐
  │  Initializer exists  │──── YES ──▶ Use initializer value
  │  for name?           │
  └─────────┬────────────┘
            │ NO
            ▼
       ┌─────────┐
       │  ERROR   │
       │ (missing │
       │  input)  │
       └─────────┘
```

In [ ]:
# Demonstrate the three resolution cases

# Case 1: Name ONLY in initializer (cannot override)
print('Case 1: A is initializer-only (frozen)')
print(f'  Graph inputs: {[i.name for i in model_init.graph.input]}')
print(f'  Initializers: {[i.name for i in model_init.graph.initializer]}')
print(f'  A is NOT in inputs → cannot be overridden at runtime')

# Case 2: Name in BOTH input and initializer (overridable default)
X2 = make_tensor_value_info('X', TensorProto.FLOAT, [None, 2])
A2 = make_tensor_value_info('A', TensorProto.FLOAT, [2, 2])  # A now also an input
Y2 = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 2])

graph2 = make_graph(
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['Y'])],
    'lr_overridable', [X2, A2], [Y2], [A_init, B_init])
model_override = make_model(graph2)
check_model(model_override)

sess2 = ort.InferenceSession(
    model_override.SerializeToString(),
    providers=['CPUExecutionProvider'])

# Use default A (from initializer)
res_default = sess2.run(None, {'X': x_test})[0]

# Override A with custom weights
A_custom = np.eye(2, dtype=np.float32)  # identity matrix
res_custom = sess2.run(None, {'X': x_test, 'A': A_custom})[0]

print('\nCase 2: A in both input AND initializer (overridable)')
print(f'  Default A result (row 0): {res_default[0]}')
print(f'  Custom  A result (row 0): {res_custom[0]}')
print(f'  Results differ: {not np.allclose(res_default, res_custom)}')

<a id='section-5'></a>
## Section 5: Attributes — Formal Definition

### Definition 5.1 (Attribute)

An **attribute** is a fixed parameter of an operator that is set at **graph construction time** and **cannot change at runtime**. Attributes are stored directly inside the `NodeProto.attribute` repeated field.

Formally, each node $v$ has:

$$v = (\text{op\_type}, \underbrace{[t_1, \ldots, t_k]}_{\text{inputs}}, \underbrace{[t'_1, \ldots, t'_l]}_{\text{outputs}}, \underbrace{\{a_1: c_1, \ldots, a_p: c_p\}}_{\text{attributes}})$$

where each attribute $a_i$ has a name and a constant value $c_i$.

### Attribute Types

| Type Code | Type Name | Example |
|:---------:|-----------|--------|
| 1 | `FLOAT` | `alpha=0.01` in LeakyRelu |
| 2 | `INT` | `axis=1` in Softmax |
| 3 | `STRING` | `mode="nearest"` in Resize |
| 4 | `TENSOR` | `value=...` in Constant |
| 5 | `GRAPH` | `then_branch=...` in If |
| 6 | `FLOATS` | `scales=[1.0, 2.0]` |
| 7 | `INTS` | `perm=[1, 0]` in Transpose |
| 8 | `STRINGS` | `keys=["a", "b"]` |

### Key Properties

1. **Immutable at runtime**: Unlike inputs (which change per inference call), attributes are baked into the graph and never change.
2. **Defined by the operator schema**: Each operator specifies which attributes it accepts, their types, and defaults.
3. **Shape-affecting**: Many attributes affect output shapes (e.g., `perm` in Transpose, `kernel_shape` in Conv).

### The Transpose Operator Example

The `Transpose` operator reverses or permutes the axes of a tensor. Its schema:

$$\text{Transpose}(T; \text{perm}) : \mathbb{R}^{d_1 \times d_2 \times \cdots \times d_r} \to \mathbb{R}^{d_{\pi(1)} \times d_{\pi(2)} \times \cdots \times d_{\pi(r)}}$$

where $\pi = \text{perm}$ is a permutation of $\{0, 1, \ldots, r-1\}$.

For a matrix ($r=2$), `perm=[1, 0]` computes the transpose:

$$A^\top_{ij} = A_{ji}$$

<a id='section-6'></a>
## Section 6: Building a Model with Attributes

Let's build a model that computes $Y = XA^\top + B$, which requires the `Transpose` operator with `perm=[1, 0]`.

### Computation Decomposition

$$\begin{align}
\text{Step 1:} \quad & T = \text{Transpose}(A; \text{perm}=[1,0]) \quad & A \in \mathbb{R}^{K \times D} \to T \in \mathbb{R}^{D \times K} \\
\text{Step 2:} \quad & H = \text{MatMul}(X, T) \quad & X \in \mathbb{R}^{N \times D}, T \in \mathbb{R}^{D \times K} \to H \in \mathbb{R}^{N \times K} \\
\text{Step 3:} \quad & Y = \text{Add}(H, B) \quad & H \in \mathbb{R}^{N \times K}, B \in \mathbb{R}^{K} \to Y \in \mathbb{R}^{N \times K}
\end{align}$$

### Graph Visualization

![Graph with Attributes](assets/dot_att.png)

Notice that `perm=[1, 0]` is an **attribute** (inside the Transpose node box), not an edge carrying a tensor.

In [ ]:
# Build the model: Y = X @ A^T + B
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

# Node 1: Transpose A with attribute perm=[1, 0]
node_transpose = make_node('Transpose', ['A'], ['tA'], perm=[1, 0])

# Node 2: MatMul(X, tA)
node_matmul = make_node('MatMul', ['X', 'tA'], ['XA'])

# Node 3: Add(XA, B)
node_add = make_node('Add', ['XA', 'B'], ['Y'])

graph = make_graph(
    [node_transpose, node_matmul, node_add],
    'lr_with_transpose',
    [X, A, B], [Y])

model_attr = make_model(graph)
check_model(model_attr)

# Inspect the Transpose node's attributes
print('Transpose node attributes:')
for attr in node_transpose.attribute:
    print(f'  name={attr.name!r}, type={attr.type}, '
          f'ints={list(attr.ints)}')

print(f'\nAll nodes:')
for n in model_attr.graph.node:
    attrs = {a.name: list(a.ints) if a.ints else a.f
             for a in n.attribute}
    print(f'  {n.op_type}: {list(n.input)} → {list(n.output)}'
          f'{" attrs=" + str(attrs) if attrs else ""}')

In [ ]:
# Verify numerically
sess_attr = ort.InferenceSession(
    model_attr.SerializeToString(),
    providers=['CPUExecutionProvider'])

x = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)  # (2, 3)
a = np.array([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]], dtype=np.float32)  # (3, 2)
b = np.array([1.0, -1.0], dtype=np.float32)  # (2,)

onnx_result = sess_attr.run(None, {'X': x, 'A': a, 'B': b})[0]
numpy_result = x @ a + b  # A is already (3,2) = (D,K), no transpose needed here

# For the ONNX model, A is transposed inside the graph
# So the equivalent NumPy is: x @ a.T + b (where a comes in as KxD)
a_transposed_input = a.T  # pretend A comes in as (2,3) and gets transposed to (3,2)
numpy_equiv = x @ a_transposed_input.T + b

print(f'ONNX result:\n{onnx_result}')
print(f'\nExpected (x @ a + b):\n{numpy_result}')
print(f'Match: {np.allclose(onnx_result, numpy_result)}')

<a id='section-7'></a>
## Section 7: Inputs vs Initializers vs Attributes — The Three-Way Taxonomy

Understanding the distinction between these three concepts is fundamental to ONNX graph construction.

### Comparison Table

| Property | **Inputs** | **Initializers** | **Attributes** |
|----------|:----------:|:----------------:|:--------------:|
| **What they are** | Tensor placeholders | Constant tensors | Operator parameters |
| **When set** | Runtime (per inference) | Model creation (fixed) | Graph creation (fixed) |
| **Can change?** | Yes, every call | No (or override if in inputs) | Never |
| **Stored in** | `GraphProto.input` | `GraphProto.initializer` | `NodeProto.attribute` |
| **Data type** | `ValueInfoProto` (type only) | `TensorProto` (type + data) | `AttributeProto` (typed value) |
| **Connected by** | Edges (SSA names) | Edges (SSA names) | Embedded in node |
| **Examples** | Input image $X$ | Weight matrix $W$ | `perm=[1,0]`, `axis=1` |

### The Data Flow Perspective

```
┌─────────────────────────────────────────────────────────────────┐
│                       GraphProto                                │
│                                                                 │
│  INPUTS (ValueInfoProto)     INITIALIZERS (TensorProto)        │
│  ┌──────────┐                ┌──────────┐                      │
│  │ X        │                │ W (data) │                      │
│  │ (shape   │                │ b (data) │                      │
│  │  only)   │                │          │                      │
│  └────┬─────┘                └────┬─────┘                      │
│       │                           │                             │
│       │    EDGES (name matching)  │                             │
│       ▼                           ▼                             │
│  ┌─────────────────────────────────────────┐                   │
│  │         NodeProto: MatMul               │                   │
│  │  inputs: ['X', 'W']                     │                   │
│  │  outputs: ['XW']                        │                   │
│  │  ATTRIBUTES: (none for MatMul)          │                   │
│  └────────────────────┬────────────────────┘                   │
│                       │                                         │
│                       ▼                                         │
│  ┌─────────────────────────────────────────┐                   │
│  │         NodeProto: Transpose            │                   │
│  │  inputs: ['XW']                         │                   │
│  │  outputs: ['XW_T']                      │                   │
│  │  ATTRIBUTES: {perm: [1, 0]}  ◀── fixed │                   │
│  └─────────────────────────────────────────┘                   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### When to Use Each

- **Input**: Data that changes every inference call (images, text, features)
- **Initializer**: Learned parameters that are fixed after training (weights, biases, embeddings)
- **Attribute**: Structural parameters that define the operator's behavior (axis, kernel size, permutation)

<a id='section-8'></a>
## Section 8: Tensor Shapes and Broadcasting

### Broadcasting Rules in ONNX

ONNX follows **NumPy broadcasting semantics**. When two tensors with different shapes are combined element-wise (Add, Mul, etc.), the shapes are aligned from the **right** and expanded:

### Rule

Given two tensors $A \in \mathbb{R}^{a_1 \times \cdots \times a_r}$ and $B \in \mathbb{R}^{b_1 \times \cdots \times b_s}$:

1. **Pad** the shorter shape with 1s on the left: if $r < s$, treat $A$ as having shape $(1, \ldots, 1, a_1, \ldots, a_r)$
2. **For each dimension** $i$, the output dimension is:

$$d_i^{\text{out}} = \begin{cases}
a_i & \text{if } b_i = 1 \\
b_i & \text{if } a_i = 1 \\
a_i & \text{if } a_i = b_i \\
\text{ERROR} & \text{otherwise}
\end{cases}$$

### Common Broadcasting Patterns

| Pattern | A Shape | B Shape | Output Shape | Use Case |
|---------|---------|---------|-------------|----------|
| Bias add | $(N, K)$ | $(K,)$ | $(N, K)$ | Adding bias to batch |
| Scalar mul | $(N, D)$ | $(1,)$ | $(N, D)$ | Scaling all elements |
| Channel-wise | $(N, C, H, W)$ | $(1, C, 1, 1)$ | $(N, C, H, W)$ | Batch normalization |

### Example: Bias Broadcasting

For $Y = XA + B$ where $X \in \mathbb{R}^{N \times D}$, $A \in \mathbb{R}^{D \times K}$, $B \in \mathbb{R}^{K}$:

$$\underbrace{XA}_{N \times K} + \underbrace{B}_{K} \xrightarrow{\text{broadcast}} \underbrace{XA}_{N \times K} + \underbrace{B}_{1 \times K} = \underbrace{Y}_{N \times K}$$

The bias vector $B$ of shape $(K,)$ is broadcast to $(1, K)$ and then replicated across all $N$ rows.

In [ ]:
# Demonstrate broadcasting with different bias shapes
test_cases = [
    ('Bias (K,)',     np.array([0.1, -0.1], dtype=np.float32)),
    ('Bias (1, K)',   np.array([[0.1, -0.1]], dtype=np.float32)),
    ('Scalar (1,)',   np.array([0.5], dtype=np.float32)),
]

x = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
a = np.array([[0.5, -0.3], [0.8, 0.2]], dtype=np.float32)
xa = x @ a

print(f'XA shape: {xa.shape}')
print(f'XA =\n{xa}\n')

for name, bias in test_cases:
    result = xa + bias
    print(f'{name:15s}  shape={str(bias.shape):10s}  →  '
          f'output shape={result.shape}  row[0]={result[0]}')

<a id='section-9'></a>
## Section 9: Numerical Walkthrough

Let's trace exact values through the initializer-based model.

### Given

**Input** (provided at runtime):
$$X = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}_{2 \times 2}$$

**Initializers** (stored in model):
$$A = \begin{bmatrix} 0.5 & -0.3 \\ 0.8 & 0.2 \end{bmatrix}_{2 \times 2}, \quad B = \begin{bmatrix} 0.1 & -0.1 \end{bmatrix}_{2}$$

### Step 1 — MatMul: $H = XA$

$$H = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix} \begin{bmatrix} 0.5 & -0.3 \\ 0.8 & 0.2 \end{bmatrix} = \begin{bmatrix} 1(0.5) + 2(0.8) & 1(-0.3) + 2(0.2) \\ 3(0.5) + 4(0.8) & 3(-0.3) + 4(0.2) \end{bmatrix} = \begin{bmatrix} 2.1 & 0.1 \\ 4.7 & -0.1 \end{bmatrix}$$

### Step 2 — Add: $Y = H + B$ (with broadcasting)

$$Y = \begin{bmatrix} 2.1 & 0.1 \\ 4.7 & -0.1 \end{bmatrix} + \underbrace{\begin{bmatrix} 0.1 & -0.1 \end{bmatrix}}_{\text{broadcast to } 2 \times 2} = \begin{bmatrix} 2.2 & 0.0 \\ 4.8 & -0.2 \end{bmatrix}$$

In [ ]:
# Verify the numerical walkthrough
x = np.array([[1, 2], [3, 4]], dtype=np.float32)

result = sess.run(None, {'X': x})[0]

step1 = x @ A_data
step2 = step1 + B_data

print('Step 1 — MatMul(X, A):')
print(f'  H = X @ A =')
print(f'  {step1}')

print('\nStep 2 — Add(H, B) with broadcast:')
print(f'  Y = H + B =')
print(f'  {step2}')

print(f'\nONNX Runtime result:')
print(f'  {result}')
print(f'\nMatch: {np.allclose(result, step2)}')

<a id='section-10'></a>
## Section 10: Visualizing the Graph

Let's create a detailed visualization showing the difference between the original (all-dynamic) graph and the initializer-based graph.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

def draw_graph(ax, title, inputs, inits, nodes, output):
    ax.set_xlim(-1, 7)
    ax.set_ylim(-1, 7)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')

    input_style = dict(boxstyle='round,pad=0.4', facecolor='#AED6F1',
                       edgecolor='#2C3E50', linewidth=2)
    init_style = dict(boxstyle='round,pad=0.4', facecolor='#D5F5E3',
                      edgecolor='#1E8449', linewidth=2)
    op_style = dict(boxstyle='square,pad=0.4', facecolor='#F9E79F',
                    edgecolor='#7D6608', linewidth=2)
    out_style = dict(boxstyle='round,pad=0.4', facecolor='#FADBD8',
                     edgecolor='#C0392B', linewidth=2)
    arrow_kw = dict(arrowstyle='->', color='#2C3E50', lw=2, mutation_scale=15)

    for name, x, y in inputs:
        ax.text(x, y, name, ha='center', va='center', fontsize=10,
                fontweight='bold', bbox=input_style)
    for name, x, y in inits:
        ax.text(x, y, name, ha='center', va='center', fontsize=10,
                fontweight='bold', bbox=init_style)
    for name, x, y in nodes:
        ax.text(x, y, name, ha='center', va='center', fontsize=11,
                fontweight='bold', bbox=op_style)
    ax.text(output[1], output[2], output[0], ha='center', va='center',
            fontsize=11, fontweight='bold', bbox=out_style)

# Left: All dynamic
draw_graph(ax1, 'Before: All Inputs Dynamic',
           inputs=[('X\n(N×D)', 1, 6), ('A\n(D×K)', 3, 6), ('B\n(K,)', 5, 4)],
           inits=[],
           nodes=[('MatMul', 2, 4), ('Add', 3, 2)],
           output=('Y\n(N×K)', 3, 0.5))
arrow_kw = dict(arrowstyle='->', color='#2C3E50', lw=2, mutation_scale=15)
ax1.annotate('', xy=(2, 4.6), xytext=(1, 5.4), arrowprops=arrow_kw)
ax1.annotate('', xy=(2, 4.6), xytext=(3, 5.4), arrowprops=arrow_kw)
ax1.annotate('', xy=(3, 2.6), xytext=(2, 3.4), arrowprops=arrow_kw)
ax1.annotate('', xy=(3, 2.6), xytext=(5, 3.4), arrowprops=arrow_kw)
ax1.annotate('', xy=(3, 1.1), xytext=(3, 1.5), arrowprops=arrow_kw)

# Right: With initializers
draw_graph(ax2, 'After: Weights as Initializers',
           inputs=[('X\n(N×D)', 1, 6)],
           inits=[('A (init)\n(D×K)', 3, 6), ('B (init)\n(K,)', 5, 4)],
           nodes=[('MatMul', 2, 4), ('Add', 3, 2)],
           output=('Y\n(N×K)', 3, 0.5))
ax2.annotate('', xy=(2, 4.6), xytext=(1, 5.4), arrowprops=arrow_kw)
ax2.annotate('', xy=(2, 4.6), xytext=(3, 5.4), arrowprops=arrow_kw)
ax2.annotate('', xy=(3, 2.6), xytext=(2, 3.4), arrowprops=arrow_kw)
ax2.annotate('', xy=(3, 2.6), xytext=(5, 3.4), arrowprops=arrow_kw)
ax2.annotate('', xy=(3, 1.1), xytext=(3, 1.5), arrowprops=arrow_kw)

legend_elements = [
    mpatches.Patch(facecolor='#AED6F1', edgecolor='#2C3E50', label='Dynamic Input'),
    mpatches.Patch(facecolor='#D5F5E3', edgecolor='#1E8449', label='Initializer (constant)'),
    mpatches.Patch(facecolor='#F9E79F', edgecolor='#7D6608', label='Operator'),
    mpatches.Patch(facecolor='#FADBD8', edgecolor='#C0392B', label='Output')]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=10)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

<a id='section-11'></a>
## Section 11: Advanced — Multiple Data Types and Initializer Inspection

### Initializers with Different Data Types

Initializers can store tensors of any ONNX element type. This is important for:
- **Quantized models**: INT8/UINT8 weights with float32 scale/zero-point
- **Index tensors**: INT64 tensors for Gather/Scatter operations
- **Boolean masks**: Bool tensors for Where operations

### Memory Impact by Data Type

For a tensor with $n$ elements:

| Data Type | Bytes per Element | Size for $n = 10^6$ |
|-----------|:-----------------:|:-------------------:|
| `FLOAT` (32-bit) | 4 | 3.81 MB |
| `FLOAT16` (16-bit) | 2 | 1.91 MB |
| `INT8` (8-bit) | 1 | 0.95 MB |
| `DOUBLE` (64-bit) | 8 | 7.63 MB |
| `INT64` (64-bit) | 8 | 7.63 MB |

In [ ]:
# Create initializers of different types
test_inits = [
    ('float32', np.array([1.0, 2.0, 3.0], dtype=np.float32)),
    ('float64', np.array([1.0, 2.0, 3.0], dtype=np.float64)),
    ('int32',   np.array([1, 2, 3], dtype=np.int32)),
    ('int64',   np.array([1, 2, 3], dtype=np.int64)),
    ('float16', np.array([1.0, 2.0, 3.0], dtype=np.float16)),
]

print(f'{"dtype":>10s} | {"ONNX type":>12s} | {"raw_bytes":>10s} | {"round-trip":>10s}')
print('-' * 55)

for name, arr in test_inits:
    proto = from_array(arr, name=f'test_{name}')
    dtype_name = TensorProto.DataType.Name(proto.data_type)
    raw_size = len(proto.raw_data)
    restored = to_array(proto)
    ok = np.array_equal(arr, restored)
    print(f'{name:>10s} | {dtype_name:>12s} | {raw_size:>10d} | {str(ok):>10s}')

In [ ]:
# Inspect initializers in our model
print('Model Initializer Report')
print('=' * 60)

total_bytes = 0
for init in model_init.graph.initializer:
    arr = to_array(init)
    dtype_name = TensorProto.DataType.Name(init.data_type)
    raw_size = len(init.raw_data)
    total_bytes += raw_size

    print(f'\n  Name:    {init.name}')
    print(f'  Shape:   {list(init.dims)}')
    print(f'  Dtype:   {dtype_name}')
    print(f'  Size:    {raw_size} bytes ({arr.size} elements × {arr.itemsize} bytes)')
    print(f'  Values:  {arr}')
    print(f'  Stats:   min={arr.min():.4f}, max={arr.max():.4f}, mean={arr.mean():.4f}')

model_bytes = len(model_init.SerializeToString())
print(f'\nTotal initializer data: {total_bytes} bytes')
print(f'Total model size:       {model_bytes} bytes')
print(f'Weight fraction:        {total_bytes/model_bytes*100:.1f}%')

<a id='section-12'></a>
## Section 12: Key Takeaways & Interview Questions

### Summary

| Concept | Key Point |
|---------|----------|
| **Initializers** | `TensorProto` objects stored in `GraphProto.initializer` that provide constant values for tensor names |
| **Attributes** | Fixed parameters inside `NodeProto` that configure operator behavior (e.g., `perm`, `axis`) |
| **Resolution** | Feed dict > Initializer > Error. Names in both inputs and initializers have overridable defaults |
| **Broadcasting** | ONNX follows NumPy rules: align shapes from the right, expand dimensions of size 1 |
| **Self-contained** | Initializers make models portable — no external weight files needed (below 2 GB) |

### Critical Rules

1. **Initializer names must match** tensor names used by nodes. The name is the "wire" that connects the stored data to the operator.

2. **Initializer-only names** (not in `graph.input`) create frozen constants. **Dual-listed names** (in both) create overridable defaults.

3. **Attributes are NOT tensors**. They are typed values (int, float, string, list) embedded in the node. They cannot be connected by edges.

4. **`from_array()` preserves dtype**. The ONNX type is determined by the NumPy array's dtype. Always use explicit dtype casting.

### Interview Questions

1. **Q**: What is the difference between an initializer and an attribute in ONNX?
   - **A**: An initializer is a constant tensor stored in the graph that flows through edges like any other tensor. An attribute is a fixed parameter embedded inside a node that configures the operator's behavior. Initializers can potentially be overridden at runtime; attributes cannot.

2. **Q**: If a tensor name appears in both `graph.input` and `graph.initializer`, what happens?
   - **A**: The initializer provides a default value. If the caller provides a value in the feed dictionary, it overrides the initializer. If not, the initializer value is used.

3. **Q**: How does ONNX handle bias addition when the bias has fewer dimensions than the matrix product?
   - **A**: ONNX uses NumPy-style broadcasting. A bias of shape $(K,)$ is broadcast to match the $(N, K)$ matrix by implicitly expanding to $(1, K)$ and replicating along the batch dimension.

4. **Q**: Why are initializers important for production deployment?
   - **A**: They make the model self-contained — the `.onnx` file includes both the computation graph and the learned weights. Without initializers, weights must be managed and transmitted separately, complicating deployment.

---

**Next:** [Opset and Metadata](../04_Opset_and_Metadata/) — Understand operator versioning and model metadata.